# OLS & (Imperfect) Multicollinearity: Monte Carlo Method
This notebook examines how multicollinearity impacts OLS estimators in two parts.

Part 1: Generates a synthetic dataset using Monte Carlo simulations.

Part 2: Analyzes how varying collinearity levels affect estimator means and standard errors.

## Part 1 - Generating the Data

Creation of the DataFrame to study the effects of (imperfect) multicollinearity on the OLS estimators.

The true population function as a DGP is assumed to be $Y_i=\beta_0+\beta X_{1i} +\beta X_{2i}$ with the true known parameters $\beta_0=2$, $\beta_1=3$ and $\beta_2=1.5$.

Based on this DGP, the Python code below uses Monte Carlo simulation to gerenerate 10,000 OLS estimators $\hat{\beta_0}$, $\hat{\beta_1}$, $\hat{\beta_2}$ for different levels of correlation between indenpendet variables $X_1$ and $X_2$. Correlation levels are: $\rho=0$, $\rho=0.25$, $\rho=0.5$, $\rho=0.75$, $\rho=0.9$ and $\rho=0.99$. Sample size in each simulation run is $N=200$.

Final results are stored in a DataFrame and exportes as csv and parquet files.

In [46]:
import numpy as np
import pandas as pd
import statsmodels.api as sm

# --- Simulation Parameters ---
N_OBS = 200         # Sample size per regression
N_SIMS = 10000      # Number of Monte Carlo iterations
BETA_0 = 2.0        # Intercept
BETA_1 = 3.0        # True coefficient for X1
BETA_2 = 1.5        # True coefficient for X2
RHO_VALUES = [0.0, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99]  # Correlation levels to test

def run_monte_carlo(rho, n_obs=N_OBS, n_sims=N_SIMS):
    b0_estimates = []
    b1_estimates = []
    b2_estimates = []

    for _ in range(n_sims):
        # 1. Generate correlated predictors X1 and X2
        cov_matrix = [[1, rho], [rho, 1]]
        X1, X2 = np.random.multivariate_normal([0, 0], cov_matrix, n_obs).T

        # 2. Generate response variable Y with normally distributed noise
        epsilon = np.random.normal(0, 1, n_obs)
        Y = BETA_0 + BETA_1 * X1 + BETA_2 * X2 + epsilon

        # 3. Fit OLS Regression
        X_design = np.column_stack([X1, X2])
        X_design = sm.add_constant(X_design)
        model = sm.OLS(Y, X_design).fit()

        # 4. Store estimated coefficients
        b0_estimates.append(model.params[0])
        b1_estimates.append(model.params[1])
        b2_estimates.append(model.params[2])

    return b0_estimates, b1_estimates, b2_estimates

# --- Store results in single DataFrames ---
my_dict={}
for index, rho in enumerate(RHO_VALUES):
    b0_hat, b1_hat, b2_hat = run_monte_carlo(rho)

    my_dict[index] = pd.DataFrame({
    'beta_0': b0_hat,
    'beta_1': b1_hat,
    'beta_2': b2_hat,
    'rho': rho
    })

# --- Combining all single DataFrames & exporting to files---
df_big=pd.concat([x for x in my_dict.values()],ignore_index=True)
df_big.to_csv("OLS_Multicollinearity.csv", index=False)
df_big.to_parquet("OLS_Multicollinearity.parquet")
